# Module 1.2 — Self-Attention, By Hand
**DeskMate SLM Curriculum · Phase 1 · Notebook 03**

Run on **Google Colab** or **Kaggle** (free T4 or CPU — both work).

By the end you will have:
- Implemented each step of scaled dot-product attention from scratch
- A working `Head` class (causal, with dropout, `register_buffer` for the mask)
- Val loss below the Module 1.1 bigram baseline
- An attention heatmap showing the causal lower-triangular structure

Read `1.2_self_attention.md` first for the theory.

---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q torch==2.3.1 matplotlib==3.9.0


In [ ]:
import random, math, pathlib, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")


## Step 1 — Runtime, Paths & Corpus

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_DIR   = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runtime : {RUNTIME}")


In [ ]:
# Load Tiny Shakespeare (download if needed)
corpus_path = DATA_DIR / "tiny_shakespeare.txt"
if not corpus_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Downloading from {url} ...")
    urllib.request.urlretrieve(url, corpus_path)

text = corpus_path.read_text(encoding="utf-8")
print(f"Corpus : {len(text):,} characters")

# Character tokenizer (same as Module 1.1)
chars = sorted(set(text))
V     = len(chars)
stoi  = {c: i for i, c in enumerate(chars)}
itos  = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"Vocab   : {V}   Train tokens: {len(train_data):,}   Val tokens: {len(val_data):,}")


In [ ]:
B, T = 32, 64   # batch size, context length

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix  = torch.randint(len(src) - T, (B,))
    x   = torch.stack([src[i   : i+T  ] for i in ix])
    y   = torch.stack([src[i+1 : i+T+1] for i in ix])
    return x.to(device), y.to(device)


## Step 2 — Attention Mechanics, Step by Step

We build scaled dot-product attention one operation at a time,
printing shapes at every step to build spatial intuition.


In [ ]:
# Toy dimensions for inspection
d_model = 32
d_head  = 16
B_toy, T_toy = 2, 8

torch.manual_seed(SEED)
x_toy = torch.randn(B_toy, T_toy, d_model)   # pretend embeddings
print(f"Input x  : {x_toy.shape}   (B, T, d_model)")

# Three projection matrices (normally learned; here random for illustration)
Wq = torch.randn(d_model, d_head)
Wk = torch.randn(d_model, d_head)
Wv = torch.randn(d_model, d_head)

Q = x_toy @ Wq    # (B, T, d_head)
K = x_toy @ Wk
V = x_toy @ Wv
print(f"Q shape  : {Q.shape}   K shape: {K.shape}   V shape: {V.shape}")


In [ ]:
# Step 1 — raw scores
scores = Q @ K.transpose(-2, -1)
print(f"scores (raw) : {scores.shape}   min={scores.min():.2f}  max={scores.max():.2f}")

# Step 2 — scale
scores_scaled = scores * d_head**-0.5
print(f"scores (scaled): min={scores_scaled.min():.2f}  max={scores_scaled.max():.2f}")
print(f"  dividing by √{d_head}={d_head**0.5:.2f} keeps scores in a stable range")


In [ ]:
# Step 3 — causal mask
tril = torch.tril(torch.ones(T_toy, T_toy))
print("Causal mask (lower-triangular):")
print(tril.int())

scores_masked = scores_scaled.masked_fill(tril == 0, float("-inf"))
print(f"\nscores[0] after masking (upper triangle is -inf):")
print(scores_masked[0].round(decimals=2))


In [ ]:
# Step 4 — softmax
weights = scores_masked.softmax(dim=-1)
print(f"weights shape : {weights.shape}")
print(f"Row sums (should all be 1.0): {weights[0].sum(dim=-1).round(decimals=4)}")
print("\nweights[0] (attention matrix for first batch item):")
print(weights[0].round(decimals=3))


In [ ]:
# Step 5 — weighted sum
out = weights @ V
print(f"output shape : {out.shape}   (B, T, d_head)")
print("Each output token is a weighted mixture of all value vectors.")


## Step 3 — The `Head` Class

Wrap the five steps into a clean `nn.Module`.
Key details: `bias=False` on projections, `register_buffer` for the mask,
slice `[:T, :T]` to support variable-length inputs.


In [ ]:
class Head(nn.Module):
    # Single causal self-attention head.

    def __init__(self, d_model: int, d_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        self.d_head = d_head
        self.q = nn.Linear(d_model, d_head, bias=False)
        self.k = nn.Linear(d_model, d_head, bias=False)
        self.v = nn.Linear(d_model, d_head, bias=False)
        self.drop = nn.Dropout(dropout)
        # Mask is not a parameter — register_buffer moves it to the right device automatically
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x)                                           # (B, T, h)
        k = self.k(x)                                           # (B, T, h)
        v = self.v(x)                                           # (B, T, h)
        scores = q @ k.transpose(-2, -1) * self.d_head**-0.5   # (B, T, T)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weights = scores.softmax(dim=-1)                        # (B, T, T)
        weights = self.drop(weights)
        return weights @ v, weights                             # (B, T, h), (B, T, T)


In [ ]:
# Shape check
d_model_real = 64
d_head_real  = 32
head = Head(d_model=d_model_real, d_head=d_head_real, block_size=T).to(device)

x_test = torch.randn(B, T, d_model_real, device=device)
out_test, w_test = head(x_test)

print(f"Input  : {x_test.shape}")
print(f"Output : {out_test.shape}   (B, T, d_head)")
print(f"Weights: {w_test.shape}   (B, T, T)")
assert out_test.shape == (B, T, d_head_real), "shape mismatch!"
assert w_test.shape  == (B, T, T),            "shape mismatch!"
print("Shape assertions passed.")


## Step 4 — Attention LM: Drop `Head` Into the Training Loop

Replace the bigram's embedding-only forward pass with:
`embed → one Head → project to vocab`. The training loop is identical to Module 1.1.


In [ ]:
D_MODEL   = 64
D_HEAD    = 32
MAX_STEPS = 3_000
LR        = 1e-3

class AttentionLM(nn.Module):
    def __init__(self, vocab_size, d_model, d_head, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.head    = Head(d_model, d_head, block_size, dropout=0.0)
        self.proj    = nn.Linear(d_head, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        tok = self.tok_emb(idx)                                        # (B, T, d_model)
        pos = self.pos_emb(torch.arange(T, device=idx.device))        # (T, d_model)
        x   = tok + pos                                                # (B, T, d_model)
        x, self._last_weights = self.head(x)                          # (B, T, d_head)
        return self.proj(x)                                            # (B, T, V)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, block_size):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits   = self(idx_cond)[:, -1, :]
            probs    = logits.softmax(dim=-1)
            nxt      = torch.multinomial(probs, 1)
            idx      = torch.cat([idx, nxt], dim=1)
        return idx


model = AttentionLM(V, D_MODEL, D_HEAD, T).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters : {n_params:,}")


In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=100):
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits = model(xb)
            losses[k] = F.cross_entropy(logits.view(-1, V), yb.view(-1))
        results[split] = losses.mean().item()
    model.train()
    return results

opt = torch.optim.AdamW(model.parameters(), lr=LR)
train_losses, val_losses, steps_logged = [], [], []

torch.manual_seed(SEED)
print(f"Initial  →  {estimate_loss()}")


In [ ]:
EVAL_INTERVAL = 500

for step in range(MAX_STEPS):
    xb, yb = get_batch("train")
    logits  = model(xb)
    loss    = F.cross_entropy(logits.view(-1, V), yb.view(-1))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        est = estimate_loss()
        train_losses.append(est["train"])
        val_losses.append(est["val"])
        steps_logged.append(step)
        print(f"step {step:5d}  train: {est['train']:.4f}  val: {est['val']:.4f}")


In [ ]:
# Compare to bigram baseline from Module 1.1
BIGRAM_VAL = 2.55   # approximate val loss at end of Module 1.1 — update if yours differs

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(steps_logged, train_losses, label="train (attention)", color="#4C72B0")
ax.plot(steps_logged, val_losses,   label="val (attention)",   color="#DD8452", linestyle="--")
ax.axhline(BIGRAM_VAL, color="grey", linestyle=":", label=f"bigram val baseline ≈ {BIGRAM_VAL}")
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("AttentionLM vs Bigram Baseline")
ax.legend(); plt.tight_layout()
plt.savefig(str(MODELS_DIR / "attention_loss_curve.png"), bbox_inches="tight")
plt.show()
print(f"Final val loss: {val_losses[-1]:.4f}  (bigram baseline: {BIGRAM_VAL})")
print(f"Improvement   : {BIGRAM_VAL - val_losses[-1]:.4f}")


## Step 5 — Attention Heatmap

Extract the `(T, T)` weight matrix from a single forward pass and plot it.
The lower-triangular structure is the causal mask in action.
Bright spots on each row reveal which past positions the model attends to most.


In [ ]:
model.eval()
sample_text = text[:T]
sample_ids  = torch.tensor([encode(sample_text)], dtype=torch.long, device=device)  # (1, T)

with torch.no_grad():
    _ = model(sample_ids)
    attn_weights = model._last_weights[0].cpu()   # (T, T) for batch item 0

print(f"Attention weights shape : {attn_weights.shape}")
print(f"Row sums (should be ~1): {attn_weights.sum(dim=-1)[:5].round(decimals=4).tolist()}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(attn_weights.numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03)

# Label axes with characters (show every 4th for readability)
tick_step = 4
tick_pos   = list(range(0, T, tick_step))
tick_chars = [sample_text[i] if sample_text[i] != "
" else "\n" for i in tick_pos]
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_chars, fontsize=8)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_chars, fontsize=8)

ax.set_xlabel("Source (key) position")
ax.set_ylabel("Query position (each row attends to columns)")
ax.set_title("Attention weights — causal mask visible as lower-triangular structure")
plt.tight_layout()
plt.savefig(str(MODELS_DIR / "attention_heatmap.png"), bbox_inches="tight")
plt.show()
model.train()


In [ ]:
# Observations
print("=== What to look for in the heatmap ===")
print()
print("1. Lower-triangular structure: row i has weight only in columns 0..i.")
print("   This is the causal mask — each token can only see the past.")
print()
print("2. Position 0 always attends only to itself (only one position available).")
print()
print("3. Some rows concentrate weight on a few positions (peaked = selective).")
print("   Others spread weight evenly (flat = uncertain or positional).")
print()
print("4. After more training, you would see structured patterns emerge:")
print("   e.g. closing quotes attending to opening quotes, newlines to newlines.")


## Checkpoint

> **Remove the causal mask — what breaks, and why?**

Write your answer below. A strong answer covers:
1. Training: what the model learns (or fails to learn) with access to future tokens.
2. Inference: why the trained model produces incoherent output at generation time.
3. Gradients: what signal the model receives about language when it can cheat.


In [ ]:
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Loss curve (attention vs bigram) | `models/attention_loss_curve.png` |
| Attention heatmap | `models/attention_heatmap.png` |

**What you've locked in:** you can now implement scaled dot-product attention from
raw matrix multiplies, apply the causal mask correctly, and read an attention heatmap.

**Next:** Module 1.3 — Multi-head attention, MLP, and the Block.
You'll run `n_heads` heads in parallel, concatenate their outputs, add a position-wise
MLP, and wrap both in residual connections + LayerNorm. The result is one stackable
`Block` — the unit repeated N times in Module 1.4 to build the full nano-SLM.
